## Imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

import mlflow
import mlflow.sklearn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

## MLFlow version 

In [3]:
import mlflow
print(mlflow.__version__)

3.14.0


In [5]:
import os

print(os.getcwd())

/home/jupyter


In [8]:
import os

os.chdir("/home/jupyter/23F1001572_MLOPS_WEEKELY_ASSIGNMENT")

print(os.getcwd())

/home/jupyter/23F1001572_MLOPS_WEEKELY_ASSIGNMENT


## Load dataset

In [12]:
df = pd.read_csv("data/train.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (120, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,4.4,2.9,1.4,0.2,setosa
1,4.9,2.5,4.5,1.7,virginica
2,6.8,2.8,4.8,1.4,versicolor
3,4.9,3.1,1.5,0.1,setosa
4,5.5,2.5,4.0,1.3,versicolor


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  120 non-null    float64
 1   sepal_width   120 non-null    float64
 2   petal_length  120 non-null    float64
 3   petal_width   120 non-null    float64
 4   species       120 non-null    object 
dtypes: float64(4), object(1)
memory usage: 4.8+ KB


## Split Dataset

In [15]:

X = df.drop("species", axis=1)
y = df["species"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training samples :", X_train.shape[0])
print("Testing samples  :", X_test.shape[0])

Training samples : 96
Testing samples  : 24


## Create an MLflow Experiment

In [16]:
import mlflow

EXPERIMENT_NAME = "iris_random_forest"

mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experiment: {EXPERIMENT_NAME}")

2026/07/22 17:27:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/22 17:27:56 INFO mlflow.store.db.utils: Updating database tables
2026/07/22 17:27:59 INFO mlflow.tracking.fluent: Experiment with name 'iris_random_forest' does not exist. Creating a new experiment.


Experiment: iris_random_forest


## Hyperparameter Grid

In [17]:
hyperparameter_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 50, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
]

hyperparameter_grid

[{'n_estimators': 50, 'max_depth': 3},
 {'n_estimators': 50, 'max_depth': 5},
 {'n_estimators': 100, 'max_depth': 3},
 {'n_estimators': 100, 'max_depth': 5}]

## Training Function

In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

best_accuracy = 0
best_model = None
best_params = None


def train_and_log(params):

    global best_accuracy
    global best_model
    global best_params

    with mlflow.start_run():

        # Train model
        model = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            random_state=42,
        )

        model.fit(X_train, y_train)

        # Predictions
        predictions = model.predict(X_test)

        # Metrics
        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(
            y_test,
            predictions,
            average="weighted",
        )
        recall = recall_score(
            y_test,
            predictions,
            average="weighted",
        )
        f1 = f1_score(
            y_test,
            predictions,
            average="weighted",
        )

        # Log parameters
        mlflow.log_params(params)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        # Log model
        mlflow.sklearn.log_model(
            sk_model=model,
            name="random_forest_model",
        )

        print(
            f"{params} -> Accuracy: {accuracy:.4f}"
        )

        # Save best model
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_model = model
            best_params = params

In [21]:
for params in hyperparameter_grid:
    train_and_log(params)

print("\nBest Accuracy :", best_accuracy)
print("Best Parameters :", best_params)

{'n_estimators': 50, 'max_depth': 3} -> Accuracy: 0.9583
{'n_estimators': 50, 'max_depth': 5} -> Accuracy: 0.9583
{'n_estimators': 100, 'max_depth': 3} -> Accuracy: 0.9583
{'n_estimators': 100, 'max_depth': 5} -> Accuracy: 0.9583

Best Accuracy : 0.9583333333333334
Best Parameters : {'n_estimators': 50, 'max_depth': 3}


In [22]:
results = []

for params in hyperparameter_grid:

    with mlflow.start_run():

        model = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            random_state=42,
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(y_test, predictions, average="weighted")
        recall = recall_score(y_test, predictions, average="weighted")
        f1 = f1_score(y_test, predictions, average="weighted")

        mlflow.log_params(params)

        mlflow.log_metrics(
            {
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
            }
        )

        mlflow.sklearn.log_model(
            sk_model=model,
            name="random_forest_model",
        )

        results.append(
            {
                **params,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
            }
        )

results_df = pd.DataFrame(results)
results_df.sort_values("accuracy", ascending=False)

,n_estimators,max_depth,accuracy,precision,recall,f1_score
0,50,3,0.958333,0.962963,0.958333,0.95817
1,50,5,0.958333,0.962963,0.958333,0.95817
2,100,3,0.958333,0.962963,0.958333,0.95817
3,100,5,0.958333,0.962963,0.958333,0.95817
